In [1]:
from os import path
import pandas as pd
import glob
import numpy as np
from IPython.display import display
from matplotlib import pyplot as plt
import napari
from skimage import io
import matplotlib.pyplot as plt
import seaborn as sns
import sys 
import zarr
import dask.array as da
import os 
pythonPackagePath = os.path.abspath(r'C:\Users\Lab admin\Desktop\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)

from detections_preprocessing import hist_plot, box_whisker_plot, plot_histogram_cutoffs
from parallel import Detector
from gaussian_visualization import visualize_3D_gaussians

plt.rcParams["font.family"] = ""

In [2]:
channel_to_detect = 3

base_dir =  r'C:\Users\Lab admin\Desktop\u-track3D\GUI_trial'

# Define the file directory and name
input_file_directory = 'OS_41_5-10min-01_processed-croppedROI/'
zarr_file_directory = input_file_directory + 'zarr_file/all_channels_data'

zarr_full_path = os.path.join(base_dir, zarr_file_directory)
z2 = zarr.open(zarr_full_path, mode='r')

In [3]:
keys = ['a1', 'a2', 'a3', 'a4']

# Get a list of .pkl files which end with _detections in C:\Users\Lab admin\Desktop\u-track3D\GUI_trial\OS_41_5-10min-01_processed-croppedROI\output\uTrack3DPackage\detection_results
pkl_files = glob.glob(r'C:\Users\Lab admin\Desktop\u-track3D\GUI_trial\OS_41_5-10min-01_processed-croppedROI\output\uTrack3DPackage\detection_results\*_detections.pkl')

# Load the .pkl files into a dictionary with key = keys
detection_results = {}
for i, file in enumerate(pkl_files):
    key = keys[i]
    detection_results[key] = pd.read_pickle(file).reset_index()

In [4]:
# Visualizing the detections in Napari

# Make a mask of the first time point of the detections

masks = []
for i in range(len(keys)):
    key = keys[i]
    df = detection_results[key]
    # Get the mask for the first time point
    mask = visualize_3D_gaussians(zarr_obj = z2, gaussians_df = df[df['frame'] == 0])
    masks.append(mask)

# Create a napari viewer
viewer = napari.Viewer()

#open the zarr file in read mode
dask_array = da.from_zarr(z2)

# first time point of the zarr file and the channel to detect
#the axis arrangement is (t,c,z,y,x)

dask_array_slice = dask_array[0,channel_to_detect-1,:,:,:]

# Add the 3D stack to the viewer
layer_raw = viewer.add_image(dask_array_slice, name='fluorescence', interpolation3d = 'nearest', blending = 'additive', colormap = 'magenta')

# layer_mask = viewer.add_image(masks, name = 'detections mask')
layer_mask0 = viewer.add_image(masks[0], name = 'a1', interpolation3d = 'nearest', blending = 'additive', colormap = 'green')
layer_mask1 = viewer.add_image(masks[1], name = 'a2', interpolation3d = 'nearest', blending = 'additive', colormap = 'red')
layer_mask2 = viewer.add_image(masks[2], name = 'a3', interpolation3d = 'nearest', blending = 'additive', colormap = 'blue')
layer_mask3 = viewer.add_image(masks[3], name = 'a4', interpolation3d = 'nearest', blending = 'additive', colormap = 'yellow')

#other useful parameters 
#color_map = list
#contrast_limits = list of list 

# Add Bounding Box
layer_raw.bounding_box.visible = True

In [5]:
# Create a napari viewer
viewer = napari.Viewer()

#access channel 3 only from zarr array 
dask_array = da.from_zarr(z2)

#the axis arrangement is (t,c,z,y,x)
# all_channels = dask_array[:,:,:,:,:]

# Import detection channel
detection_channel = dask_array[:, int(channel_to_detect)-1,:,:,:]


# Add the 4D stack to the viewer
# layer_raw = viewer.add_image(all_channels, channel_axis = 1, name = ['Channel 1', 'Channel 2', 'Channel 3'])
# layer_raw = viewer.add_image(all_channels, channel_axis = 1, name = ['channel 1', 'channel 2', 'channel 3'], interpolation3d = 'nearest', blending = 'additive', colormap = 'gray_r', visible = [False, False, True])

layer_raw = viewer.add_image(detection_channel, name = 'fluorescence', interpolation3d = 'nearest', blending = 'additive', colormap = 'gray_r', visible = True)

#other useful parameters 
#color_map = list
#contrast_limits = list of list 

# Add Bounding Box
layer_raw.bounding_box.visible = True
# layer_raw[0].bounding_box.visible = True
# layer_raw[1].bounding_box.visible = True
# layer_raw[2].bounding_box.visible = True

#Visualising all dropped spots and the cleaned spots 
points_layer = viewer.add_points(detection_results[keys[0]][["frame", "mu_z", "mu_y", "mu_x"]], size=3, 
                                name = 'a1', face_color = 'lime', symbol = 'ring')


points_layer = viewer.add_points(detection_results[keys[1]][["frame", "mu_z", "mu_y", "mu_x"]], size=3, 
                                name = 'a2', face_color = 'orange', symbol = 'ring')

points_layer = viewer.add_points(detection_results[keys[2]][["frame", "mu_z", "mu_y", "mu_x"]], size=3,
                                name = 'a3', face_color = 'purple', symbol = 'ring')

points_layer = viewer.add_points(detection_results[keys[3]][["frame", "mu_z", "mu_y", "mu_x"]], size=3,
                                name = 'a4', face_color = 'red', symbol = 'ring')